# OASIS-1: how much does the MRI add beyond age, sex and education? (first training notebook)

**Before you run:** right side panel -> *Add Input* -> your private `oasis1-prepped` dataset. *Settings* -> Accelerator **GPU T4 x2** (one GPU is used; the model is small) and **Internet ON** (needed once to download ImageNet-pretrained weights).

**Run the cells one by one, top to bottom.** Cell 4 draws pictures: look at them. Cell 7 is a 1-3 minute smoke test: if it fails, copy the red error to Claude. Only then run Cell 8 (the real run).

Rules built in: splits are by subject (never by slice); MMSE and CDR are never used as inputs; the number of epochs is fixed in advance, so nothing is tuned on test data.

In [ ]:
# ---- Cell 1: setup and settings ------------------------------------------------
import os, glob, json, time, math, random, warnings
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

INPUT_ROOT = os.environ.get('INPUT_ROOT', '/kaggle/input')
WORK = os.environ.get('WORK_DIR', '/kaggle/working')

CFG = dict(
    TASKS=['dementia'],     # 'dementia' = CDR 0 vs CDR>0 ; 'cdr05' = CDR 0 vs CDR 0.5 (CDR>=1 dropped)
    VIEWS=['axial', 'coronal', 'sagittal'],
    N_SLICES=12,            # slices per view per subject
    CENTRAL_FRAC=0.4,       # use the central 40% of the brain along each axis
    STEP=2,                 # neighbouring-slice offset for the 3-channel (2.5D) input
    IMG=192, EPOCHS=12, BATCH=64, LR=2e-4,
    REPEATS=5,              # 1 = first look (about 10 min per task); 5 = final numbers
    SEED=42)
VIEW_AXIS = {'axial': 2, 'coronal': 1, 'sagittal': 0}   # checked visually in Cell 5

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device, '| GPUs:', torch.cuda.device_count(), '| torch', torch.__version__)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
set_seed(CFG['SEED'])

In [ ]:
# ---- Cell 2: find the data and the subject tables --------------------------------
# normal run (ages 60+, 198 subjects):
SUBJECTS_FILE, SPLITS_FILE, RUN_TAG = 'subjects.csv', 'splits.csv', ''
# ALL-AGES run (ages 18+, 235 subjects): remove the '#' on the next line
# SUBJECTS_FILE, SPLITS_FILE, RUN_TAG = 'subjects_allages.csv', 'splits_allages.csv', '_allages'
npz_files = sorted(glob.glob(f'{INPUT_ROOT}/**/*.npz', recursive=True))
subj_csv  = sorted(glob.glob(f'{INPUT_ROOT}/**/{SUBJECTS_FILE}', recursive=True))
split_csv = sorted(glob.glob(f'{INPUT_ROOT}/**/{SPLITS_FILE}', recursive=True))
print(len(npz_files), 'volumes |', subj_csv, split_csv)
assert npz_files and subj_csv and split_csv, 'Missing files: add the dataset with the volumes AND the tables (' + SUBJECTS_FILE + ', ' + SPLITS_FILE + ')'

subjects_all = pd.read_csv(subj_csv[0])
splits = pd.read_csv(split_csv[0])
path_of = {os.path.basename(f)[:-4]: f for f in npz_files}          # 'OAS1_0001_MR1' -> file
subjects_all['path'] = subjects_all['session_id'].map(path_of)
print('subjects in table:', len(subjects_all), '| with a volume:', subjects_all.path.notna().sum())
subjects_all = subjects_all.dropna(subset=['path']).reset_index(drop=True)
print(subjects_all.groupby('CDR').size().rename('subjects per CDR').to_string())

def get_task(task):
    d = subjects_all.copy()
    if task == 'dementia':
        d['y'] = (d.CDR > 0).astype(int)
    elif task == 'cdr05':
        d = d[d.CDR.isin([0, 0.5])].copy(); d['y'] = (d.CDR == 0.5).astype(int)
    else:
        raise ValueError(task)
    return d.reset_index(drop=True)

def make_folds(d, repeats):
    # (repeat, fold, train_idx, test_idx) over rows of d, using the saved subject-level splits
    pos = {s: i for i, s in enumerate(d.subject_id)}
    out = []
    for r in range(repeats):
        sp = splits[splits['repeat'] == r]
        for k in sorted(sp['test_fold'].unique()):
            te = np.array(sorted(pos[s] for s in sp[sp['test_fold'] == k]['subject_id'] if s in pos))
            tr = np.array(sorted(set(range(len(d))) - set(te)))
            if len(te) and len(set(d.y.values[te])) == 2:
                out.append((r, int(k), tr, te))
    return out

In [ ]:
# ---- Cell 3: cut each brain volume into 2.5D slices (3 views) ---------------------------
def view_slices(vol, axis, n, frac, step):
    nz = vol != 0
    other = tuple(a for a in range(3) if a != axis)
    idx = np.where(nz.sum(axis=other) > 50)[0]
    lo, hi = int(idx[0]), int(idx[-1])
    c, half = (lo + hi) / 2.0, frac * (hi - lo) / 2.0
    pos = np.linspace(c - half, c + half, n).round().astype(int)
    size = vol.shape[axis]; out = []
    for p in pos:
        ch = np.clip([p - step, p, p + step], 0, size - 1)
        out.append(np.moveaxis(np.take(vol, ch, axis=axis), axis, 0))   # (3, H, W)
    return np.stack(out).astype(np.float16)                              # (n, 3, H, W)

def build_samples(d):
    X = {v: None for v in CFG['VIEWS']}
    t0 = time.time()
    for i, f in enumerate(d.path):
        vol = np.load(f)['vol'].astype(np.float32)
        for v in CFG['VIEWS']:
            s = view_slices(vol, VIEW_AXIS[v], CFG['N_SLICES'], CFG['CENTRAL_FRAC'], CFG['STEP'])
            if X[v] is None:
                X[v] = np.zeros((len(d),) + s.shape, dtype=np.float16)
            X[v][i] = s
        if (i + 1) % 50 == 0: print(f'  {i+1}/{len(d)} volumes cut ({time.time()-t0:.0f}s)')
    return X

# all subjects once; every task re-uses these arrays
X_all = build_samples(subjects_all)
for v in CFG['VIEWS']:
    print(v, X_all[v].shape, f'{X_all[v].nbytes/1e9:.2f} GB')

In [ ]:
# ---- Cell 4: LOOK at the pictures (3 subjects x 3 views). Do the names match what you see? ---
import matplotlib.pyplot as plt
picks = []
for c in [0, 0.5, 1]:
    m = subjects_all.index[subjects_all.CDR == c]
    if len(m): picks.append(m[0])
fig, ax = plt.subplots(len(picks), 3, figsize=(10, 3.4 * len(picks)))
ax = np.atleast_2d(ax)
for r, i in enumerate(picks):
    for cidx, v in enumerate(CFG['VIEWS']):
        ax[r, cidx].imshow(X_all[v][i, CFG['N_SLICES'] // 2, 1].astype(np.float32).T, cmap='gray', origin='lower')
        ax[r, cidx].set_title(f"{v} | CDR {subjects_all.CDR[i]:g} | age {subjects_all.age[i]:g}", fontsize=9)
        ax[r, cidx].axis('off')
plt.tight_layout(); plt.show()
print('axial = looking down from the top (two hemispheres side by side); coronal = front view; sagittal = side view.')

In [ ]:
# ---- Figure 1: sample slices. Rows: OASIS-1 control, OASIS-1 dementia, OASIS-2 control. Columns: axial, coronal, sagittal ----
# Needs: Cells 1, 2, 3 run (60+ line active in Cell 2) and the 'oasis2-prepped' dataset added as input.
import matplotlib
import matplotlib.pyplot as plt

s1 = subjects_all[subjects_all.age >= 60]
def closest(df, target=76):
    return (df.age - target).abs().idxmin()          # subject whose age is nearest to 76
i_ctl = closest(s1[s1.CDR == 0])
i_dem = closest(s1[s1.CDR >= 1])                     # a clear dementia case (CDR 1)

s2_csv = sorted(glob.glob(f'{INPUT_ROOT}/**/subjects_oasis2.csv', recursive=True))
n2_files = sorted(glob.glob(f'{INPUT_ROOT}/**/prepped/OAS2_*_MR1.npz', recursive=True))
assert s2_csv and len(n2_files) == 150, 'add the oasis2-prepped dataset to this notebook'
d2f = pd.read_csv(s2_csv[0])
d2f['path'] = d2f.session_id.map({os.path.basename(f)[:-4]: f for f in n2_files})
j = closest(d2f[d2f.CDR == 0])
vol2 = np.load(d2f.path[j])['vol'].astype(np.float32)

mid = CFG['N_SLICES'] // 2
def one_view(row, v):
    if row == 2:
        sl = view_slices(vol2, VIEW_AXIS[v], CFG['N_SLICES'], CFG['CENTRAL_FRAC'], CFG['STEP'])
        return sl[mid, 1].astype(np.float32)
    i = i_ctl if row == 0 else i_dem
    return X_all[v][i, mid, 1].astype(np.float32)

labels = [f"OASIS-1 control\n(CDR 0, {subjects_all.age[i_ctl]:g} y)",
          f"OASIS-1 dementia\n(CDR {subjects_all.CDR[i_dem]:g}, {subjects_all.age[i_dem]:g} y)",
          f"OASIS-2 control\n(CDR 0, {d2f.age[j]:g} y)"]
fig, ax = plt.subplots(3, 3, figsize=(6.2, 6.4))
for r in range(3):
    for c, v in enumerate(CFG['VIEWS']):
        ax[r, c].imshow(one_view(r, v).T, cmap='gray', origin='lower')
        ax[r, c].set_xticks([]); ax[r, c].set_yticks([])
        if r == 0: ax[r, c].set_title(v.capitalize(), fontsize=10)
        if c == 0: ax[r, c].set_ylabel(labels[r], fontsize=8)
plt.tight_layout()
plt.savefig('/kaggle/working/fig_samples.png', dpi=300, bbox_inches='tight')
plt.show()
print('saved /kaggle/working/fig_samples.png')
print('OASIS-1 control :', subjects_all.subject_id[i_ctl], '| OASIS-1 dementia:', subjects_all.subject_id[i_dem], '| OASIS-2:', d2f.subject_id[j])

In [ ]:
# ---- Cell 5: baselines WITHOUT images (age, age+sex+education) on the same folds ------------
def oof_covariates(d, cols, repeats):
    P = np.full((repeats, len(d)), np.nan)
    X = d[cols].copy(); X = X.fillna(X.median()).values.astype(float)
    for r, k, tr, te in make_folds(d, repeats):
        m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
        m.fit(X[tr], d.y.values[tr]); P[r, te] = m.predict_proba(X[te])[:, 1]
    return P

def boot_ci(y, p_avg, n=1000, seed=0, q=(2.5, 97.5)):
    rng = np.random.default_rng(seed); a = []
    for _ in range(n):
        i = rng.integers(0, len(y), len(y))
        if len(set(y[i])) == 2: a.append(roc_auc_score(y[i], p_avg[i]))
    return np.percentile(a, q)

COV = {}
for task in CFG['TASKS']:
    d = get_task(task); y = d.y.values
    print(f'\nTASK {task}: n={len(d)} (negative {int((y==0).sum())}, positive {int((y==1).sum())})')
    for name, cols in [('age only', ['age']), ('age+sex+educ', ['age', 'sex', 'educ'])]:
        P = oof_covariates(d, cols, CFG['REPEATS']); COV[(task, name)] = P
        pa = np.nanmean(P, 0); lo, hi = boot_ci(y, pa)
        aucs = [roc_auc_score(y, P[r]) for r in range(CFG['REPEATS'])]
        print(f'  {name:14s} AUC {np.mean(aucs):.3f}  (95% CI {lo:.3f}-{hi:.3f})')

In [ ]:
# ---- Cell 6: model and training code ---------------------------------------------------------
SAVE_AS = None   # if set to a file path, run_fold_view saves the trained model there

def make_model():
    try:
        m = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.DEFAULT)
    except Exception as e:
        print('! pretrained weights not available (turn Internet ON). Using random weights.', repr(e)[:120])
        m = torchvision.models.resnet18(weights=None)
    m.fc = nn.Linear(512, 1)
    return m

def to_in(xb):
    xb = xb.to(device, non_blocking=True).float()
    return F.interpolate(xb, size=(CFG['IMG'], CFG['IMG']), mode='bilinear', align_corners=False)

def augment(x):
    B, dev = x.size(0), x.device
    ang = (torch.rand(B, device=dev) * 2 - 1) * math.radians(6)
    sc  = 1 + (torch.rand(B, device=dev) * 2 - 1) * 0.05
    tx  = (torch.rand(B, device=dev) * 2 - 1) * 0.05
    ty  = (torch.rand(B, device=dev) * 2 - 1) * 0.05
    cs, sn = torch.cos(ang) / sc, torch.sin(ang) / sc
    theta = torch.stack([torch.stack([cs, -sn, tx], 1), torch.stack([sn, cs, ty], 1)], 1)   # (B, 2, 3)
    grid = F.affine_grid(theta, list(x.shape), align_corners=False)
    return F.grid_sample(x, grid, padding_mode='zeros', align_corners=False)

def run_fold_view(X, y, tr_idx, te_idx, seed, epochs):
    # X: (Nsubj, S, 3, H, W) float16 ; returns one probability per test subject
    set_seed(seed)
    S = X.shape[1]
    Xtr = torch.from_numpy(X[tr_idx].reshape((-1,) + X.shape[2:]))
    ytr = torch.from_numpy(np.repeat(y[tr_idx], S)).float()
    model = make_model().to(device)
    pos_w = torch.tensor(float((ytr == 0).sum()) / max(float((ytr == 1).sum()), 1.0), device=device)
    opt = torch.optim.AdamW(model.parameters(), lr=CFG['LR'], weight_decay=1e-2)
    steps = epochs * math.ceil(len(Xtr) / CFG['BATCH'])
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=CFG['LR'], total_steps=steps, pct_start=0.2)
    scaler = torch.cuda.amp.GradScaler(enabled=(device == 'cuda'))
    for ep in range(epochs):
        model.train(); perm = torch.randperm(len(Xtr))
        for i in range(0, len(perm), CFG['BATCH']):
            b = perm[i:i + CFG['BATCH']]
            if len(b) < 2: continue
            xb = augment(to_in(Xtr[b])); yb = ytr[b].to(device)
            with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=(device == 'cuda')):
                out = model(xb).squeeze(1)
            loss = F.binary_cross_entropy_with_logits(out.float(), yb, pos_weight=pos_w)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
    model.eval()
    Xte = torch.from_numpy(X[te_idx].reshape((-1,) + X.shape[2:]))
    probs = []
    with torch.no_grad():
        for i in range(0, len(Xte), 128):
            with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=(device == 'cuda')):
                o = model(to_in(Xte[i:i + 128])).squeeze(1)
            probs.append(torch.sigmoid(o.float()).cpu())
    p = torch.cat(probs).numpy().reshape(len(te_idx), S).mean(1)      # average the slices of each subject
    if SAVE_AS:
        os.makedirs(os.path.dirname(SAVE_AS), exist_ok=True)
        torch.save(model.state_dict(), SAVE_AS)
    del model, Xtr; torch.cuda.empty_cache()
    return p

In [ ]:
# ---- Cell 7: SMOKE TEST (1-3 minutes). If this fails, send me the error. ------------------------
t0 = time.time()
d = get_task('dementia'); y = d.y.values
sel = np.r_[np.where(y == 0)[0][:16], np.where(y == 1)[0][:16]]
tr, te = sel[::2], sel[1::2]
for v in CFG['VIEWS']:
    p = run_fold_view(X_all[v], y, tr, te, seed=0, epochs=1)
    print(f'{v:9s} ok | probs {np.round(p[:4], 2)} | {time.time()-t0:.0f}s elapsed')
print('SMOKE TEST PASSED. Now run Cell 8.')

In [ ]:
# ---- Cell 8: THE REAL RUN (cross-validation, subject-level folds) ---------------------------------
RESULTS = {}
for task in CFG['TASKS']:
    d = get_task(task); y = d.y.values
    idx = subjects_all.reset_index().set_index('subject_id').loc[d.subject_id, 'index'].values   # rows of X_all
    folds = make_folds(d, CFG['REPEATS'])
    P = {v: np.full((CFG['REPEATS'], len(d)), np.nan) for v in CFG['VIEWS']}
    same = len(idx) == len(subjects_all) and bool((idx == np.arange(len(idx))).all())
    XV = {v: (X_all[v] if same else X_all[v][idx]) for v in CFG['VIEWS']}
    t0 = time.time()
    for n, (r, k, tr, te) in enumerate(folds):
        for v in CFG['VIEWS']:
            P[v][r, te] = run_fold_view(XV[v], y, tr, te, seed=1000 * r + k, epochs=CFG['EPOCHS'])
        print(f'[{task}] repeat {r} fold {k} done ({n+1}/{len(folds)}) | {time.time()-t0:.0f}s')
    P['ensemble'] = np.nanmean(np.stack([P[v] for v in CFG['VIEWS']]), 0)
    RESULTS[task] = dict(d=d, y=y, P=P)
    rows = []
    for name, arr in P.items():
        pa = np.nanmean(arr, 0); lo, hi = boot_ci(y, pa)
        aucs = [roc_auc_score(y, arr[r]) for r in range(CFG['REPEATS'])]
        pred = (pa >= 0.5).astype(int)
        sens = ((pred == 1) & (y == 1)).sum() / (y == 1).sum(); spec = ((pred == 0) & (y == 0)).sum() / (y == 0).sum()
        rows.append(dict(model=name, auc=np.mean(aucs), ci_low=lo, ci_high=hi, sens=sens, spec=spec, bal_acc=(sens + spec) / 2))
    for cname in ['age only', 'age+sex+educ']:
        pa = np.nanmean(COV[(task, cname)], 0); lo, hi = boot_ci(y, pa)
        rows.append(dict(model=cname + ' (no images)', auc=roc_auc_score(y, pa), ci_low=lo, ci_high=hi))
    tab = pd.DataFrame(rows).round(3)
    print(f'\n===== {task} =====\n', tab.to_string(index=False))
    # does the image ensemble beat age+sex+education? (paired bootstrap over subjects)
    pe, pc = np.nanmean(P['ensemble'], 0), np.nanmean(COV[(task, 'age+sex+educ')], 0)
    rng = np.random.default_rng(1); diffs = []
    for _ in range(1000):
        i = rng.integers(0, len(y), len(y))
        if len(set(y[i])) == 2: diffs.append(roc_auc_score(y[i], pe[i]) - roc_auc_score(y[i], pc[i]))
    print(f'AUC gain of images over age+sex+educ: {np.mean(diffs):+.3f}  (95% CI {np.percentile(diffs, 2.5):+.3f} to {np.percentile(diffs, 97.5):+.3f})')
    tab.to_csv(f'{WORK}/results{RUN_TAG}_{task}.csv', index=False)
    out = d[['subject_id', 'CDR', 'age', 'sex', 'educ', 'y']].copy()
    for name, arr in P.items(): out['p_' + name] = np.nanmean(arr, 0)
    out.to_csv(f'{WORK}/oof_predictions{RUN_TAG}_{task}.csv', index=False)
print('\nSaved results_*.csv and oof_predictions_*.csv in /kaggle/working (Output tab).')

## Part 2: explainability and age (cells 9-12)

**After a fresh start** (new session) run Cells 1, 2, 3, 5 and 6 first (Cell 4, 7 and 8 can be skipped).

* **Cell 9** (about 2 min, Internet ON): downloads a standard brain atlas and lines it up with our brains. Check the picture: white outline = ventricles, yellow = hippocampus.
* **Cell 10** (about 12 min): retrains the 5 folds of repeat 0, then measures where the network looks and whether that attention is trustworthy (region overlap, deletion/insertion tests against random pixels, randomisation check).
* **Cell 10b** (10 s): redraws the attention picture with better contrast; no retraining.
* **Cell 11** (about 35 min per task with REPEATS = 5): network that receives age as an extra input. In our run this gave no gain over the image-only network (a reportable negative result).
* **Cell 12** (1 min, CPU): age calibration as a second stage on top of the image score, evaluated with cross-validation. Needs `oof_predictions_<task>.csv` from Cell 8 (in /kaggle/working or in an input dataset).

In [ ]:
# ---- Cell 9: brain-region masks (Harvard-Oxford atlas) lined up with OUR volumes -----------------
# Needs Internet ON (downloads the atlas once). Takes about 1-2 minutes. Run Cells 1-3 first.
import itertools, subprocess, sys

def load_ref(d, n=25, seed=0):
    """Average brain of n subjects (used only to line the atlas up) and the brain outline."""
    rng = np.random.default_rng(seed)
    ids = rng.choice(len(d), size=min(n, len(d)), replace=False)
    s, cnt = None, None
    for i in ids:
        v = np.load(d.path[i])['vol'].astype(np.float32)
        s = v if s is None else s + v
        cnt = (v != 0).astype(np.float32) if cnt is None else cnt + (v != 0)
    return s / len(ids), (cnt / len(ids)) > 0.5

def load_atlas():
    try:
        from nilearn import datasets
    except Exception:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'nilearn'], check=False)
        from nilearn import datasets
    import nibabel as nib
    ho = datasets.fetch_atlas_harvard_oxford('sub-maxprob-thr25-1mm')
    img = ho['maps']
    if isinstance(img, (str, os.PathLike)): img = nib.load(img)
    return np.asarray(img.get_fdata()).astype(np.int16), list(ho['labels'])

def label_ids(labels, word):
    return [i for i, l in enumerate(labels) if word.lower() in str(l).lower() and i > 0]

def orient_arr(A, perm, flips):
    B = np.transpose(A, perm)
    for ax, f in enumerate(flips):
        if f: B = np.flip(B, ax)
    return B

def bbox(mask):
    idx = np.argwhere(mask)
    return idx.min(0), idx.max(0)

def _sub(p, n=1500):
    if len(p) <= n: return p.astype(float)
    return p[np.random.default_rng(0).choice(len(p), n, replace=False)].astype(float)

def align_atlas(A, vent_ids, other_ids, ref, brain, step=3, span=12):
    """Try all 48 axis orders/flips + shifts; keep the one where the atlas ventricles land on dark CSF
    in our average brain and the other regions land inside the brain. Returns the transform."""
    shape = np.array(ref.shape)
    blo, bhi = bbox(brain); co = (blo + bhi) / 2.0; ext_o = (bhi - blo + 1).astype(float)
    shifts = np.array(list(itertools.product(range(-span, span + 1, step), repeat=3)), dtype=float)
    best = None
    for perm in itertools.permutations(range(3)):
        for flips in itertools.product([False, True], repeat=3):
            B = orient_arr(A, perm, flips)
            alo, ahi = bbox(B > 0); ca = (alo + ahi) / 2.0; ext_a = (ahi - alo + 1).astype(float)
            k = float(np.median(ext_o / ext_a))
            vp = np.argwhere(np.isin(B, vent_ids)); op = np.argwhere(np.isin(B, other_ids))
            if len(vp) < 200 or len(op) < 200: continue
            vb = (_sub(vp) - ca) * k + co; ob = (_sub(op) - ca) * k + co
            for t in shifts:
                q = np.rint(vb + t).astype(int)
                ok = ((q >= 0) & (q < shape)).all(1)
                if ok.mean() < 0.9: continue
                z = ref[q[ok, 0], q[ok, 1], q[ok, 2]].mean()
                q2 = np.rint(ob + t).astype(int)
                ok2 = ((q2 >= 0) & (q2 < shape)).all(1)
                inside = np.zeros(len(q2), bool)
                inside[ok2] = brain[q2[ok2, 0], q2[ok2, 1], q2[ok2, 2]]
                sc = z + 3.0 * (1.0 - inside.mean())
                if best is None or sc < best[0]:
                    best = (sc, perm, flips, ca, k, co, t.copy(), z, inside.mean())
    return best

def mask_in_ours(B, ids, ca, k, co, t, shape):
    ax = [np.rint((np.arange(shape[a]) - co[a] - t[a]) / k + ca[a]).astype(int) for a in range(3)]
    okk = [(x >= 0) & (x < B.shape[a]) for a, x in enumerate(ax)]
    axc = [np.clip(x, 0, B.shape[a] - 1) for a, x in enumerate(ax)]
    m = np.isin(B[np.ix_(*axc)], ids)
    return m & okk[0][:, None, None] & okk[1][None, :, None] & okk[2][None, None, :]

def build_rois(A, labels, ref, brain):
    hip = label_ids(labels, 'hippocampus'); ven = label_ids(labels, 'lateral ventricle'); amy = label_ids(labels, 'amygdala')
    print('atlas label ids  hippocampus', hip, '| lateral ventricle', ven, '| amygdala', amy)
    best = align_atlas(A, ven, hip + amy, ref, brain)
    if best is None: raise RuntimeError('alignment search found nothing')
    sc, perm, flips, ca, k, co, t, z_vent, inside = best
    B = orient_arr(A, perm, flips)
    masks = {'hippocampus': mask_in_ours(B, hip, ca, k, co, t, ref.shape),
             'ventricles': mask_in_ours(B, ven, ca, k, co, t, ref.shape)}
    masks['medial_temporal'] = masks['hippocampus'] | mask_in_ours(B, amy, ca, k, co, t, ref.shape)
    info = dict(axis_order=perm, flips=flips, scale=round(k, 3), shift=t.tolist(), vent_mean_z=float(z_vent), medial_inside_brain=float(inside),
                hip_mean_z=float(ref[masks['hippocampus']].mean()), vox={n: int(m.sum()) for n, m in masks.items()})
    return masks, info

ROI_MASKS, ROI_OK = None, False
try:
    d_ref = get_task('dementia')
    REF, BRAIN = load_ref(d_ref)
    ATLAS, ATLAS_LABELS = load_atlas()
    ROI_MASKS, ROI_INFO = build_rois(ATLAS, ATLAS_LABELS, REF, BRAIN)
    print('\nalignment:', ROI_INFO)
    ROI_OK = (ROI_INFO['vent_mean_z'] < -0.3) and (ROI_INFO['medial_inside_brain'] > 0.9)
    print('\nALIGNMENT CHECK:', 'OK (ventricle region is dark CSF and the other regions are inside the brain)' if ROI_OK
          else 'NOT RELIABLE - ROI numbers will be skipped. Send Claude the picture below.')
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(1, 3, figsize=(12, 4.2))
    for a_i, v in enumerate(CFG['VIEWS']):
        axis = VIEW_AXIS[v]; c = int(np.mean([BRAIN.nonzero()[axis].min(), BRAIN.nonzero()[axis].max()]))
        ax[a_i].imshow(np.take(REF, c, axis=axis).T, cmap='gray', origin='lower')
        for nm, col, ls in [('ventricles', 'white', '-'), ('hippocampus', 'yellow', '-')]:
            m = np.take(ROI_MASKS[nm], c, axis=axis).T
            if m.any(): ax[a_i].contour(m.astype(float), levels=[0.5], colors=col, linewidths=1.2)
        ax[a_i].set_title(f'{v}: white = ventricles, yellow = hippocampus', fontsize=9); ax[a_i].axis('off')
    plt.tight_layout(); plt.savefig(f'{WORK}/roi_alignment.png', dpi=150); plt.show()
    print('Look at the picture: white should sit on the dark ventricles, yellow on the hippocampus (a small curved structure next to the ventricle horns).')
except Exception as e:
    print('! Could not build the region masks:', repr(e)[:300], '\nThe rest of the explainability tests still run; only the region-overlap part is skipped.')

In [ ]:
# ---- Cell 10: EXPLAINABILITY - where does the model look, and can we trust it? -----------------
# Retrains the 5 folds of repeat 0 (about 7 min), then for every held-out subject measures:
#   (a) how strongly the Grad-CAM attention sits on the hippocampus / ventricles (enrichment: 1 = chance)
#   (b) deletion / insertion tests: do the highlighted pixels really drive the prediction?  (vs random pixels)
#   (c) a randomisation check: does the map change when the network's weights are scrambled?
# Run Cells 1-3, 6 and 9 first (Cell 5 is not needed).
import copy, scipy.stats as st
import matplotlib.pyplot as plt

XAI_TASK, XAI_REPEAT = 'dementia', 0
STEPS = [0.0, 0.05, 0.10, 0.20, 0.30, 0.50]       # share of brain pixels removed / revealed
N_RAND_SUBJ = 15                                  # subjects used for the randomisation check
trapz = getattr(np, 'trapezoid', None) or np.trapz
if 'ROI_OK' not in globals(): ROI_MASKS, ROI_OK = None, False
GN = ['CDR 0', 'CDR 0.5', 'CDR>=1']
grp = lambda c: 0 if c == 0 else (1 if c == 0.5 else 2)

def train_view_model(X, y, tr_idx, seed, epochs):
    set_seed(seed)
    S = X.shape[1]
    Xtr = torch.from_numpy(X[tr_idx].reshape((-1,) + X.shape[2:]))
    ytr = torch.from_numpy(np.repeat(y[tr_idx], S)).float()
    model = make_model().to(device)
    pos_w = torch.tensor(float((ytr == 0).sum()) / max(float((ytr == 1).sum()), 1.0), device=device)
    opt = torch.optim.AdamW(model.parameters(), lr=CFG['LR'], weight_decay=1e-2)
    steps = epochs * math.ceil(len(Xtr) / CFG['BATCH'])
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=CFG['LR'], total_steps=steps, pct_start=0.2)
    scaler = torch.cuda.amp.GradScaler(enabled=(device == 'cuda'))
    for ep in range(epochs):
        model.train(); perm = torch.randperm(len(Xtr))
        for i in range(0, len(perm), CFG['BATCH']):
            b = perm[i:i + CFG['BATCH']]
            if len(b) < 2: continue
            xb = augment(to_in(Xtr[b])); yb = ytr[b].to(device)
            with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=(device == 'cuda')):
                out = model(xb).squeeze(1)
            loss = F.binary_cross_entropy_with_logits(out.float(), yb, pos_weight=pos_w)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
    del Xtr
    return model.eval()

def gradcam(model, x, sign=1.0):
    """Grad-CAM on the last conv block. sign=+1: evidence FOR dementia; sign=-1: evidence AGAINST (for the control class)."""
    store = {}
    h = model.layer4.register_forward_hook(lambda m, i, o: store.__setitem__('a', o))
    try:
        with torch.enable_grad():
            out = model(x).squeeze(1)
            a = store['a']
            g = torch.autograd.grad((sign * out).sum(), a)[0]
    finally:
        h.remove()
    w = g.mean((2, 3), keepdim=True)
    cam = F.relu((w * a).sum(1, keepdim=True))
    cam = F.interpolate(cam, size=x.shape[-2:], mode='bilinear', align_corners=False)[:, 0]
    return cam.detach(), torch.sigmoid(out.detach().float())

@torch.no_grad()
def mean_prob(model, x):
    return torch.sigmoid(model(x).squeeze(1).float()).mean().item()

def del_ins_curves(model, x, cam):
    """x: (S,3,H,W) one subject's slices; cam: (S,H,W) evidence for the class the model predicted.
    Deletion: replace the most-attended pixels by a blurred copy of the image, step by step. Insertion: start from the
    blurred image and reveal them. The same with RANDOM pixels is the control."""
    S, _, H, W = x.shape
    brain = (x[:, 1] != 0)
    nb = int(brain.sum())
    sc = cam.clone(); sc[~brain] = -1.0
    order = {'cam': torch.argsort(sc.flatten(), descending=True)}
    rr = torch.rand(S * H * W, device=x.device); rr[~brain.flatten()] = -1.0
    order['rand'] = torch.argsort(rr, descending=True)
    blur = F.avg_pool2d(x, 21, stride=1, padding=10)
    res = {}
    for name, od in order.items():
        dele, ins = [], []
        for f in STEPS:
            m = torch.zeros(S * H * W, dtype=torch.bool, device=x.device)
            m[od[:int(f * nb)]] = True
            m = m.view(S, 1, H, W)
            dele.append(mean_prob(model, torch.where(m, blur, x)))
            ins.append(mean_prob(model, torch.where(m, x, blur)))
        res[name] = (np.array(dele), np.array(ins))
    return res

def slice_pos(vol, axis, n, frac):                 # same rule as Cell 3
    nz = vol != 0
    other = tuple(a for a in range(3) if a != axis)
    idx = np.where(nz.sum(axis=other) > 50)[0]
    lo, hi = int(idx[0]), int(idx[-1])
    c, half = (lo + hi) / 2.0, frac * (hi - lo) / 2.0
    return np.linspace(c - half, c + half, n).round().astype(int)

def to_volume(cam_n, pos, axis, shape):
    v3 = np.zeros(shape, np.float32); cov = np.zeros(shape, bool)
    t = np.moveaxis(v3, axis, 0); c = np.moveaxis(cov, axis, 0)
    for j, p in enumerate(pos):
        t[p] = np.maximum(t[p], cam_n[j]); c[p] = True
    return v3, cov

def enrichment(v3, cov, brain, roi):
    m = cov & brain
    tot = float(v3[m].sum()); frac_vox = float((m & roi).sum()) / max(float(m.sum()), 1.0)
    if tot <= 0 or frac_vox <= 0: return np.nan
    return (float(v3[m & roi].sum()) / tot) / frac_vox

def reinit(model, parts):
    m = copy.deepcopy(model)
    for name in parts:
        for mod in getattr(m, name).modules():
            if hasattr(mod, 'reset_parameters'): mod.reset_parameters()
    return m.eval()

def rand_model():
    m = torchvision.models.resnet18(weights=None); m.fc = nn.Linear(512, 1)
    return m.to(device).eval()

def cam_corr(c1, c2):
    a = F.avg_pool2d(c1[:, None], 4)[:, 0].flatten().cpu().numpy(); b = F.avg_pool2d(c2[:, None], 4)[:, 0].flatten().cpu().numpy()
    return st.spearmanr(a, b)[0] if a.std() > 0 and b.std() > 0 else np.nan

def draw_attention(MEAN, path):
    """Average attention maps on the middle slice, with contrast set from the data (no retraining needed)."""
    from scipy.ndimage import gaussian_filter
    fig, ax = plt.subplots(4, 3, figsize=(11, 14))
    for vi, v in enumerate(CFG['VIEWS']):
        axis = VIEW_AXIS[v]; nzc = BRAIN.nonzero()[axis]; c = int((nzc.min() + nzc.max()) // 2)
        bg = np.take(REF, c, axis=axis).T; inside = bg != 0
        sl = {g: gaussian_filter(np.take(MEAN[(v, g)].astype(np.float32), c, axis=axis).T, 1.5) for g in range(3) if (v, g) in MEAN}
        vmax = max(float(np.percentile(s_[inside], 99.5)) for s_ in sl.values()) + 1e-8
        for gi in range(3):
            a = ax[gi, vi]; a.axis('off')
            if gi not in sl: continue
            a.imshow(bg, cmap='gray', origin='lower')
            a.imshow(np.ma.masked_less(sl[gi] / vmax, 0.15), cmap='inferno', alpha=0.75, origin='lower', vmin=0, vmax=1)
            if ROI_OK:
                for nm, col in [('ventricles', 'white'), ('hippocampus', 'cyan')]:
                    m = np.take(ROI_MASKS[nm], c, axis=axis).T
                    if m.any(): a.contour(m.astype(float), levels=[0.5], colors=col, linewidths=0.9)
            a.set_title(f'{v} | {GN[gi]} | n={int(((R.view == v) & (R.group == gi)).sum())}', fontsize=9)
        a = ax[3, vi]; a.axis('off')
        if 0 in sl and 2 in sl:
            dif = sl[2] - sl[0]; lim = float(np.percentile(np.abs(dif[inside]), 99)) + 1e-8
            a.imshow(bg, cmap='gray', origin='lower')
            a.imshow(np.ma.masked_where(np.abs(dif) < 0.2 * lim, dif), cmap='coolwarm', alpha=0.8, origin='lower', vmin=-lim, vmax=lim)
            a.set_title(f'{v}: CDR>=1 minus CDR 0 (red = more attention in dementia)', fontsize=8)
    plt.suptitle('Average Grad-CAM evidence for dementia, middle slice (bright = more).  white = ventricles, cyan = hippocampus', fontsize=10)
    plt.tight_layout(); plt.savefig(path, dpi=150); plt.show()

# ------------------------------- main loop -------------------------------------
d = get_task(XAI_TASK); y = d.y.values
if 'REF' not in globals(): REF, BRAIN = load_ref(d)
idx = subjects_all.reset_index().set_index('subject_id').loc[d.subject_id, 'index'].values
same = len(idx) == len(subjects_all) and bool((idx == np.arange(len(idx))).all())
XV = {v: (X_all[v] if same else X_all[v][idx]) for v in CFG['VIEWS']}
folds = [f for f in make_folds(d, XAI_REPEAT + 1) if f[0] == XAI_REPEAT]
ACC = {}            # (view, group) -> [sum of CAM volumes, coverage count]
ROWS, CURVES, RAND = [], [], []
t0 = time.time()
for n, (r, k, tr, te) in enumerate(folds):
    models = {v: train_view_model(XV[v], y, tr, seed=1000 * r + k, epochs=CFG['EPOCHS']) for v in CFG['VIEWS']}
    print(f'fold {k}: models trained | {time.time()-t0:.0f}s')
    for i in te:
        vol = np.load(d.path[i])['vol'].astype(np.float32); brainv = vol != 0
        for v in CFG['VIEWS']:
            axis = VIEW_AXIS[v]; model = models[v]
            x = to_in(torch.from_numpy(XV[v][i]))
            cam, pr = gradcam(model, x, 1.0)                       # evidence FOR dementia -> region maps
            p0 = pr.mean().item()
            camp = cam if p0 >= 0.5 else gradcam(model, x, -1.0)[0]   # evidence for the PREDICTED class -> deletion/insertion
            cam = cam / cam.max().clamp_min(1e-8); camp = camp / camp.max().clamp_min(1e-8)
            row = dict(subject_id=d.subject_id[i], CDR=d.CDR[i], group=grp(d.CDR[i]), y=int(y[i]), view=v, p=p0, pred=int(p0 >= 0.5))
            # (a) where does the attention sit?
            pos = slice_pos(vol, axis, CFG['N_SLICES'], CFG['CENTRAL_FRAC'])
            da, db = XV[v].shape[-2:]
            cam_n = F.interpolate(cam[:, None], size=(da, db), mode='bilinear', align_corners=False)[:, 0].cpu().numpy()
            if n == 0 and i == te[0]:           # guard: our slice positions really are the ones used for training
                chk = np.moveaxis(np.take(vol, pos, axis=axis), axis, 0)
                assert np.array_equal(chk.astype(np.float16), XV[v][i][:, 1]), 'slice positions do not match Cell 3'
            v3, cov = to_volume(cam_n, pos, axis, vol.shape)
            g = row['group']
            if (v, g) not in ACC: ACC[(v, g)] = [np.zeros(vol.shape, np.float32), np.zeros(vol.shape, np.float32)]
            ACC[(v, g)][0] += v3; ACC[(v, g)][1] += cov
            if ROI_OK:
                for nm, roi in ROI_MASKS.items(): row['enr_' + nm] = enrichment(v3, cov, brainv, roi)
            # (b) deletion / insertion against random pixels, on the model's predicted class
            res = del_ins_curves(model, x, camp)
            flip = (lambda a: a) if p0 >= 0.5 else (lambda a: 1.0 - a)
            for name, (dl, ins) in res.items():
                dl, ins = flip(dl), flip(ins)
                row[f'del_auc_{name}'] = float(trapz(dl, STEPS) / (STEPS[-1] - STEPS[0]))
                row[f'ins_auc_{name}'] = float(trapz(ins, STEPS) / (STEPS[-1] - STEPS[0]))
                row[f'drop20_{name}'] = float(dl[0] - dl[3])
                CURVES.append(dict(view=v, kind=name, dele=dl, ins=ins))
            ROWS.append(row)
    # (c) randomisation check on the first fold only
    if n == 0:
        set_seed(123)
        for v in CFG['VIEWS']:
            rp, rf = reinit(models[v], ['layer4', 'fc']), rand_model()
            for i in te[:N_RAND_SUBJ]:
                x = to_in(torch.from_numpy(XV[v][i]))
                c0, _ = gradcam(models[v], x); c1, _ = gradcam(rp, x); c2, _ = gradcam(rf, x)
                RAND.append(dict(view=v, trained_vs_top_layers_scrambled=cam_corr(c0, c1), trained_vs_fully_random=cam_corr(c0, c2)))
    del models; torch.cuda.empty_cache()
    print(f'fold {k} explained ({n+1}/{len(folds)}) | {time.time()-t0:.0f}s')

R = pd.DataFrame(ROWS)
R.to_csv(f'{WORK}/xai_per_subject.csv', index=False)

def bci(x, n=2000, seed=0):
    x = np.asarray(x, float); x = x[~np.isnan(x)]
    if len(x) == 0: return (np.nan, np.nan, np.nan)
    rng = np.random.default_rng(seed); bm = [x[rng.integers(0, len(x), len(x))].mean() for _ in range(n)]
    return (x.mean(), *np.percentile(bm, [2.5, 97.5]))
fmt = lambda t: f'{t[0]:.2f} ({t[1]:.2f}-{t[2]:.2f})'

# ---- (a) region overlap -------------------------------------------------------------------------
if ROI_OK:
    out = []
    for v in CFG['VIEWS'] + ['all views']:
        for gi, gname in enumerate(GN + ['all subjects']):
            sub = R[(R.view == v) if v != 'all views' else R.view.notna()]
            if gname != 'all subjects': sub = sub[sub.group == gi]
            if len(sub) == 0: continue
            rec = dict(view=v, group=gname, n=len(sub))
            for nm in ROI_MASKS: rec[nm] = fmt(bci(sub['enr_' + nm]))
            out.append(rec)
    E = pd.DataFrame(out); E.to_csv(f'{WORK}/xai_region_enrichment.csv', index=False)
    print('\n=== (a) REGION ENRICHMENT: share of attention inside the region / share of brain volume it occupies ===')
    print('    1.0 = attention spread like chance; >1 = concentrated on the region.   mean (95% CI)\n', E.to_string(index=False))
else:
    print('\n(a) skipped: region masks were not reliable (see Cell 9).')

# ---- (b) deletion / insertion -------------------------------------------------------------------
def faith_table(sub, label):
    out = []
    for v in CFG['VIEWS']:
        s_ = sub[sub.view == v]
        if len(s_) < 5: continue
        out.append(dict(subjects=label, view=v, n=len(s_),
            deletion_CAM=fmt(bci(s_.del_auc_cam)), deletion_random=fmt(bci(s_.del_auc_rand)), deletion_CAM_minus_random=fmt(bci(s_.del_auc_cam - s_.del_auc_rand)),
            insertion_CAM=fmt(bci(s_.ins_auc_cam)), insertion_random=fmt(bci(s_.ins_auc_rand)), insertion_CAM_minus_random=fmt(bci(s_.ins_auc_cam - s_.ins_auc_rand)),
            drop20_CAM=fmt(bci(s_.drop20_cam)), drop20_random=fmt(bci(s_.drop20_rand))))
    return out
FA = pd.DataFrame(faith_table(R, 'all') + faith_table(R[R.pred == 1], 'predicted dementia') + faith_table(R[R.pred == 0], 'predicted control'))
FA.to_csv(f'{WORK}/xai_faithfulness.csv', index=False)
print('\n=== (b) FAITHFULNESS: probability of the model\'s predicted class while the most-attended (CAM) or random pixels are blurred out / revealed ===')
print('    deletion: CAM minus random should be NEGATIVE (removing attended pixels hurts more). insertion: CAM minus random should be POSITIVE.')
for lab in FA.subjects.unique(): print('\n--', lab); print(FA[FA.subjects == lab].drop(columns='subjects').set_index('view').T.to_string())

# ---- (c) randomisation sanity check -------------------------------------------------------------
RA = pd.DataFrame(RAND).groupby('view').agg(lambda c: fmt(bci(c))); RA.to_csv(f'{WORK}/xai_randomisation.csv')
print('\n=== (c) RANDOMISATION CHECK: rank correlation of the map with the map from a scrambled network ===')
print('    a trustworthy explanation CHANGES when the weights are scrambled (correlation near 0).\n', RA.to_string())

# ---- pictures -----------------------------------------------------------------------------------
MEAN = {}
for (v, gi), (sm, cv) in ACC.items(): MEAN[(v, gi)] = (sm / np.maximum(cv, 1)).astype(np.float16)
np.savez_compressed(f'{WORK}/xai_mean_attention_volumes.npz', **{f'{v}_{GN[gi].replace(" ", "")}': m for (v, gi), m in MEAN.items()})
draw_attention(MEAN, f'{WORK}/xai_mean_attention.png')

fig, ax = plt.subplots(2, 3, figsize=(11, 6))
for vi, v in enumerate(CFG['VIEWS']):
    for ri, key in enumerate(['dele', 'ins']):
        for kind, ls in [('cam', '-'), ('rand', '--')]:
            M = np.mean([c[key] for c in CURVES if c['view'] == v and c['kind'] == kind], 0)
            ax[ri, vi].plot(np.array(STEPS) * 100, M, ls, color='k', marker='o', ms=3, label='Grad-CAM order' if kind == 'cam' else 'random order')
        ax[ri, vi].set_title(f'{v}: {"deletion" if key == "dele" else "insertion"}', fontsize=10)
        ax[ri, vi].set_xlabel('% of brain pixels removed' if key == 'dele' else '% of brain pixels revealed'); ax[ri, vi].set_ylabel('prob. of predicted class')
        ax[ri, vi].legend(fontsize=8)
plt.tight_layout(); plt.savefig(f'{WORK}/xai_faithfulness.png', dpi=150); plt.show()
print('\nSaved in /kaggle/working: xai_per_subject.csv, xai_region_enrichment.csv, xai_faithfulness.csv, xai_randomisation.csv, xai_*.png, roi_alignment.png')

In [ ]:
# ---- Cell 10b: redraw the attention picture with better contrast (no retraining, 10 seconds) ------
# Run right after Cell 10 in the SAME session. Needs Cell 9 (REF, ROI_MASKS) and Cell 10 (MEAN, R) in memory.
import matplotlib.pyplot as plt
GN = ['CDR 0', 'CDR 0.5', 'CDR>=1']
if 'MEAN' not in globals():
    z = np.load(f'{WORK}/xai_mean_attention_volumes.npz')
    MEAN = {(k.split('_')[0], GN.index(k.split('_', 1)[1].replace('CDR', 'CDR '))): z[k] for k in z.files}
def draw_attention(MEAN, path):
    """Average attention maps on the middle slice, with contrast set from the data (no retraining needed)."""
    from scipy.ndimage import gaussian_filter
    fig, ax = plt.subplots(4, 3, figsize=(11, 14))
    for vi, v in enumerate(CFG['VIEWS']):
        axis = VIEW_AXIS[v]; nzc = BRAIN.nonzero()[axis]; c = int((nzc.min() + nzc.max()) // 2)
        bg = np.take(REF, c, axis=axis).T; inside = bg != 0
        sl = {g: gaussian_filter(np.take(MEAN[(v, g)].astype(np.float32), c, axis=axis).T, 1.5) for g in range(3) if (v, g) in MEAN}
        vmax = max(float(np.percentile(s_[inside], 99.5)) for s_ in sl.values()) + 1e-8
        for gi in range(3):
            a = ax[gi, vi]; a.axis('off')
            if gi not in sl: continue
            a.imshow(bg, cmap='gray', origin='lower')
            a.imshow(np.ma.masked_less(sl[gi] / vmax, 0.15), cmap='inferno', alpha=0.75, origin='lower', vmin=0, vmax=1)
            if ROI_OK:
                for nm, col in [('ventricles', 'white'), ('hippocampus', 'cyan')]:
                    m = np.take(ROI_MASKS[nm], c, axis=axis).T
                    if m.any(): a.contour(m.astype(float), levels=[0.5], colors=col, linewidths=0.9)
            a.set_title(f'{v} | {GN[gi]} | n={int(((R.view == v) & (R.group == gi)).sum())}', fontsize=9)
        a = ax[3, vi]; a.axis('off')
        if 0 in sl and 2 in sl:
            dif = sl[2] - sl[0]; lim = float(np.percentile(np.abs(dif[inside]), 99)) + 1e-8
            a.imshow(bg, cmap='gray', origin='lower')
            a.imshow(np.ma.masked_where(np.abs(dif) < 0.2 * lim, dif), cmap='coolwarm', alpha=0.8, origin='lower', vmin=-lim, vmax=lim)
            a.set_title(f'{v}: CDR>=1 minus CDR 0 (red = more attention in dementia)', fontsize=8)
    plt.suptitle('Average Grad-CAM evidence for dementia, middle slice (bright = more).  white = ventricles, cyan = hippocampus', fontsize=10)
    plt.tight_layout(); plt.savefig(path, dpi=150); plt.show()

draw_attention(MEAN, f'{WORK}/xai_mean_attention.png')

In [ ]:
import glob
for pat in ['/kaggle/working/**/xai_mean_attention_volumes.npz', '/kaggle/input/**/xai_mean_attention_volumes.npz']:
    print(pat, '->', glob.glob(pat, recursive=True))

In [ ]:
# ---- Cell 10c: redraw the CDR>=1 attention maps at the SAME slice level the network sees (about 10 seconds) ----
# Needs: Cells 1, 2, 3 and 9 run in this session (REF, BRAIN, ROI_MASKS come from Cell 9) and the file
# /kaggle/working/xai_mean_attention_volumes.npz saved by Cell 10.
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter

z = np.load(f'{WORK}/xai_mean_attention_volumes.npz')
print('keys in file:', z.files)
GROUP = 'CDR>=1'

def central_index(mask, axis, n=12, frac=0.4):
    # same rule as the training slices: brain extent along the axis, middle of the central 40 %
    other = tuple(a for a in range(3) if a != axis)
    idx = np.where(mask.sum(axis=other) > 50)[0]
    lo, hi = int(idx[0]), int(idx[-1])
    c, half = (lo + hi) / 2.0, frac * (hi - lo) / 2.0
    return int(np.linspace(c - half, c + half, n).round().astype(int)[n // 2])

fig, ax = plt.subplots(1, 3, figsize=(7.2, 2.7))
for i, v in enumerate(CFG['VIEWS']):
    axis = VIEW_AXIS[v]
    c = central_index(BRAIN, axis)
    bg = np.take(REF, c, axis=axis).T
    att = gaussian_filter(np.take(z[f'{v}_{GROUP}'].astype(np.float32), c, axis=axis).T, 1.5)
    vmax = float(np.percentile(att[bg != 0], 99.5)) + 1e-8
    ax[i].imshow(bg, cmap='gray', origin='lower')
    ax[i].imshow(np.ma.masked_less(att / vmax, 0.15), cmap='inferno', alpha=0.75, origin='lower', vmin=0, vmax=1)
    if ROI_OK:
        for nm, col in [('ventricles', 'white'), ('hippocampus', 'yellow')]:
            m = np.take(ROI_MASKS[nm], c, axis=axis).T
            if m.any(): ax[i].contour(m.astype(float), levels=[0.5], colors=col, linewidths=0.9)
    ax[i].axis('off')
    print(v, 'slice index', c)
plt.tight_layout(pad=0.3)
plt.savefig(f'{WORK}/xai_cdr1_central.png', dpi=300, bbox_inches='tight')
plt.show()
print('saved', f'{WORK}/xai_cdr1_central.png')

In [ ]:
# ---- Cell 10d: clean version of the CDR>=1 attention figure (colour only inside the brain, tight crop, white background) ----
# Needs: Cells 1, 2, 3 and 9 run in this session (REF, BRAIN, ROI_MASKS) and xai_mean_attention_volumes.npz from Cell 10.
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter, binary_fill_holes

z = np.load(f'{WORK}/xai_mean_attention_volumes.npz')
GROUP = 'CDR>=1'

def central_index(mask, axis, n=12, frac=0.4):
    other = tuple(a for a in range(3) if a != axis)
    idx = np.where(mask.sum(axis=other) > 50)[0]
    lo, hi = int(idx[0]), int(idx[-1])
    c, half = (lo + hi) / 2.0, frac * (hi - lo) / 2.0
    return int(np.linspace(c - half, c + half, n).round().astype(int)[n // 2])

gray = plt.get_cmap('gray').copy(); gray.set_bad('white')
fig, ax = plt.subplots(1, 3, figsize=(7.2, 2.7))
for i, v in enumerate(CFG['VIEWS']):
    axis = VIEW_AXIS[v]
    c = central_index(BRAIN, axis)
    sl = lambda A: np.take(A, c, axis=axis).T
    brain = binary_fill_holes(sl(BRAIN))
    bg = np.where(brain, sl(REF), np.nan)
    att = gaussian_filter(sl(z[f'{v}_{GROUP}'].astype(np.float32)), 1.5)
    vmax = float(np.percentile(att[brain], 99.5)) + 1e-8
    ax[i].imshow(bg, cmap=gray, origin='lower')
    ax[i].imshow(np.ma.masked_where(~brain, np.clip(att / vmax, 0, 1)), cmap='inferno', alpha=0.6, origin='lower', vmin=0, vmax=1)
    if ROI_OK:
        for nm, col in [('ventricles', 'white'), ('hippocampus', 'yellow')]:
            m = sl(ROI_MASKS[nm])
            if m.any(): ax[i].contour(m.astype(float), levels=[0.5], colors=col, linewidths=0.9)
    ys, xs = np.where(brain); pad = 4
    ax[i].set_xlim(xs.min() - pad, xs.max() + pad); ax[i].set_ylim(ys.min() - pad, ys.max() + pad)
    ax[i].axis('off')
    print(v, 'slice index', c)
plt.tight_layout(pad=0.3)
plt.savefig(f'{WORK}/xai_cdr1_clean.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print('saved', f'{WORK}/xai_cdr1_clean.png')

In [ ]:
# ---- Cell 11: AGE-AWARE variant - the network also receives the person's age ---------------------
# Same folds, same seeds, same epochs as Cell 8. The only change: age (scaled with training-fold statistics)
# is joined to the image features before the final layers, so the network can judge atrophy *for that age*.
# Sex, education, MMSE and CDR are NOT inputs. Run Cells 1-3, 5, 6 first. Set REPEATS = 5 in Cell 1 so it is
# comparable with your 5-repeat result (about 35 min per task).
class AgeNet(nn.Module):
    def __init__(self):
        super().__init__()
        base = make_model(); base.fc = nn.Identity(); self.base = base
        self.head = nn.Sequential(nn.Linear(512 + 1, 64), nn.ReLU(), nn.Dropout(0.2), nn.Linear(64, 1))
    def forward(self, x, age):
        return self.head(torch.cat([self.base(x), age[:, None]], 1)).squeeze(1)

def run_fold_view_age(X, y, age, tr_idx, te_idx, seed, epochs):
    set_seed(seed)
    S = X.shape[1]
    mu, sd = float(age[tr_idx].mean()), float(age[tr_idx].std()) + 1e-6
    agez = ((age - mu) / sd).astype(np.float32)
    Xtr = torch.from_numpy(X[tr_idx].reshape((-1,) + X.shape[2:]))
    ytr = torch.from_numpy(np.repeat(y[tr_idx], S)).float()
    atr = torch.from_numpy(np.repeat(agez[tr_idx], S))
    model = AgeNet().to(device)
    pos_w = torch.tensor(float((ytr == 0).sum()) / max(float((ytr == 1).sum()), 1.0), device=device)
    opt = torch.optim.AdamW(model.parameters(), lr=CFG['LR'], weight_decay=1e-2)
    steps = epochs * math.ceil(len(Xtr) / CFG['BATCH'])
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=CFG['LR'], total_steps=steps, pct_start=0.2)
    scaler = torch.cuda.amp.GradScaler(enabled=(device == 'cuda'))
    for ep in range(epochs):
        model.train(); perm = torch.randperm(len(Xtr))
        for i in range(0, len(perm), CFG['BATCH']):
            b = perm[i:i + CFG['BATCH']]
            if len(b) < 2: continue
            xb = augment(to_in(Xtr[b])); yb = ytr[b].to(device); ab = atr[b].to(device)
            with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=(device == 'cuda')):
                out = model(xb, ab)
            loss = F.binary_cross_entropy_with_logits(out.float(), yb, pos_weight=pos_w)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
    model.eval()
    Xte = torch.from_numpy(X[te_idx].reshape((-1,) + X.shape[2:]))
    ate = torch.from_numpy(np.repeat(agez[te_idx], S))
    probs = []
    with torch.no_grad():
        for i in range(0, len(Xte), 128):
            with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=(device == 'cuda')):
                o = model(to_in(Xte[i:i + 128]), ate[i:i + 128].to(device))
            probs.append(torch.sigmoid(o.float()).cpu())
    p = torch.cat(probs).numpy().reshape(len(te_idx), S).mean(1)
    del model, Xtr; torch.cuda.empty_cache()
    return p

AGE_RES = {}
for task in CFG['TASKS']:
    d = get_task(task); y = d.y.values; age = d.age.values.astype(float)
    idx = subjects_all.reset_index().set_index('subject_id').loc[d.subject_id, 'index'].values
    same = len(idx) == len(subjects_all) and bool((idx == np.arange(len(idx))).all())
    XV = {v: (X_all[v] if same else X_all[v][idx]) for v in CFG['VIEWS']}
    folds = make_folds(d, CFG['REPEATS'])
    P = {v: np.full((CFG['REPEATS'], len(d)), np.nan) for v in CFG['VIEWS']}
    t0 = time.time()
    for n, (r, k, tr, te) in enumerate(folds):
        for v in CFG['VIEWS']:
            P[v][r, te] = run_fold_view_age(XV[v], y, age, tr, te, seed=1000 * r + k, epochs=CFG['EPOCHS'])
        print(f'[{task}, age-aware] repeat {r} fold {k} done ({n+1}/{len(folds)}) | {time.time()-t0:.0f}s')
    P['ensemble'] = np.nanmean(np.stack([P[v] for v in CFG['VIEWS']]), 0)
    pa = {nm: np.nanmean(a, 0) for nm, a in P.items()}
    # image-only predictions for the paired comparison
    if 'RESULTS' in globals() and task in RESULTS:
        plain = {nm: np.nanmean(a, 0) for nm, a in RESULTS[task]['P'].items()}; src = 'Cell 8 (this session)'
    else:
        cands = glob.glob(f'{WORK}/oof_predictions_{task}.csv') + glob.glob(f'{INPUT_ROOT}/**/oof_predictions_{task}.csv', recursive=True)
        if cands:
            o = pd.read_csv(cands[0]).set_index('subject_id').loc[d.subject_id]
            plain = {nm: o['p_' + nm].values for nm in ['axial', 'coronal', 'sagittal', 'ensemble']}; src = cands[0]
        else:
            plain, src = None, None
    cov = np.nanmean(COV[(task, 'age+sex+educ')], 0)
    rows = []
    for nm in CFG['VIEWS'] + ['ensemble']:
        lo, hi = boot_ci(y, pa[nm])
        rows.append(dict(model=nm + ' (age-aware)', auc=np.mean([roc_auc_score(y, P[nm][r]) for r in range(CFG['REPEATS'])]), ci_low=lo, ci_high=hi))
    if plain is not None:
        lo, hi = boot_ci(y, plain['ensemble'])
        rows.append(dict(model='ensemble (image only, Cell 8)', auc=roc_auc_score(y, plain['ensemble']), ci_low=lo, ci_high=hi))
    lo, hi = boot_ci(y, cov); rows.append(dict(model='age+sex+educ (no images)', auc=roc_auc_score(y, cov), ci_low=lo, ci_high=hi))
    tab = pd.DataFrame(rows).round(3)
    print(f'\n===== {task}: age-aware network =====\n', tab.to_string(index=False))
    if plain is not None: print('(image-only predictions taken from:', src, ')')
    rng = np.random.default_rng(2)
    def paired(a, b, name):
        df = []
        for _ in range(1000):
            i = rng.integers(0, len(y), len(y))
            if len(set(y[i])) == 2: df.append(roc_auc_score(y[i], a[i]) - roc_auc_score(y[i], b[i]))
        print(f'{name}: {np.mean(df):+.3f}  (95% CI {np.percentile(df, 2.5):+.3f} to {np.percentile(df, 97.5):+.3f})')
    paired(pa['ensemble'], cov, 'age-aware ensemble minus age+sex+educ')
    if plain is not None:
        paired(pa['ensemble'], plain['ensemble'], 'age-aware ensemble minus image-only ensemble')
        bands = pd.cut(age, [59, 69, 79, 100], labels=['60-69', '70-79', '80+'])
        print('\nAUC by age band (ensemble):   age-aware | image-only')
        for b in ['60-69', '70-79', '80+']:
            m = np.asarray(bands == b)
            if m.sum() > 5 and len(set(y[m])) == 2:
                print(f'   {b:6s} n={int(m.sum()):3d}   {roc_auc_score(y[m], pa["ensemble"][m]):.3f} | {roc_auc_score(y[m], plain["ensemble"][m]):.3f}')
        from scipy.stats import spearmanr
        print('score-vs-age correlation among controls (lower = less age shortcut):  age-aware %.2f | image-only %.2f' %
              (spearmanr(pa['ensemble'][y == 0], age[y == 0])[0], spearmanr(plain['ensemble'][y == 0], age[y == 0])[0]))
    tab.to_csv(f'{WORK}/results_age_{task}.csv', index=False)
    out = d[['subject_id', 'CDR', 'age', 'sex', 'educ', 'y']].copy()
    for nm, a in pa.items(): out['p_' + nm] = a
    out.to_csv(f'{WORK}/oof_predictions_age_{task}.csv', index=False)
    AGE_RES[task] = pa
print('\nSaved results_age_*.csv and oof_predictions_age_*.csv in /kaggle/working (Output tab).')

In [ ]:
# ---- Cell 12: second-stage AGE CALIBRATION of the image score (CPU only, 1 minute) ----------------
# Idea: the same atrophy score means something different at 65 and at 85. A tiny logistic model takes the
# image score (from Cell 8, out-of-fold) and age, and is itself evaluated with cross-validation.
# Needs the file oof_predictions_<task>.csv from Cell 8 (in /kaggle/working, or in an input dataset you uploaded).
from sklearn.model_selection import StratifiedKFold
from scipy.stats import spearmanr

def _cv_pred(X, y, R=50):
    X = (X - X.mean(0)) / X.std(0); P = np.zeros((R, len(y)))
    for r in range(R):
        for tr, te in StratifiedKFold(5, shuffle=True, random_state=r).split(X, y):
            P[r, te] = LogisticRegression().fit(X[tr], y[tr]).predict_proba(X[te])[:, 1]
    return P

_lg = lambda p: np.log(np.clip(p, 1e-4, 1 - 1e-4) / (1 - np.clip(p, 1e-4, 1 - 1e-4)))
for task in CFG['TASKS']:
    cands = glob.glob(f'{WORK}/oof_predictions_{task}.csv') + glob.glob(f'{INPUT_ROOT}/**/oof_predictions_{task}.csv', recursive=True)
    if not cands: print(f'[{task}] no oof_predictions_{task}.csv found (run Cell 8 or add it as an input dataset)'); continue
    a = pd.read_csv(cands[0]); y = a.y.values; age = a.age.values.astype(float); c = a.CDR.values
    Pi = _cv_pred(np.c_[_lg(a.p_ensemble)], y); Ps = _cv_pred(np.c_[_lg(a.p_ensemble), age], y)
    Pc = _cv_pred(a[['age', 'sex', 'educ']].values.astype(float), y)
    pi, ps, pc = Pi.mean(0), Ps.mean(0), Pc.mean(0)
    auc = lambda P: np.mean([roc_auc_score(y, p) for p in P])
    Z = np.c_[_lg(a.p_ensemble), age]; coef = LogisticRegression().fit((Z - Z.mean(0)) / Z.std(0), y).coef_[0]
    print(f'\n===== {task}: image score + age (second stage), read from {cands[0]} =====')
    print(f'AUC  image only {auc(Pi):.3f} | image + age {auc(Ps):.3f} | age+sex+educ {auc(Pc):.3f}')
    print(f'standardised weights: image score {coef[0]:+.2f}, age {coef[1]:+.2f}   (negative age weight = older people need a higher score to be called demented)')
    rng = np.random.default_rng(0)
    def pb(x, z, B=2000):
        d_ = []
        for _ in range(B):
            i = rng.integers(0, len(y), len(y))
            if len(set(y[i])) == 2: d_.append(roc_auc_score(y[i], x[i]) - roc_auc_score(y[i], z[i]))
        return f'{np.mean(d_):+.3f} (95% CI {np.percentile(d_, 2.5):+.3f} to {np.percentile(d_, 97.5):+.3f})'
    print('gain of image+age over image only :', pb(ps, pi)); print('gain of image+age over age+sex+educ:', pb(ps, pc))
    bands = pd.cut(age, [59, 69, 79, 100], labels=['60-69', '70-79', '80+'])
    print('AUC by age band      image only | image + age')
    for b in ['60-69', '70-79', '80+']:
        m = np.asarray(bands == b)
        if m.sum() > 5 and len(set(y[m])) == 2: print(f'   {b:6s} n={int(m.sum()):3d}   {roc_auc_score(y[m], pi[m]):.3f} | {roc_auc_score(y[m], ps[m]):.3f}')
    if task == 'dementia':
        for nm, mk in [('CDR 0.5 vs 0', c <= 0.5), ('CDR>=1 vs 0', (c == 0) | (c >= 1))]:
            print(f'   {nm:13s}      {roc_auc_score((c[mk] > 0).astype(int), pi[mk]):.3f} | {roc_auc_score((c[mk] > 0).astype(int), ps[mk]):.3f}')
    pd.DataFrame(dict(subject_id=a.subject_id, y=y, p_image=pi, p_image_age=ps)).to_csv(f'{WORK}/oof_predictions_agecal_{task}.csv', index=False)

In [ ]:
# ---- Cell 13: EXTERNAL TEST. Train on all 198 OASIS-1 subjects (60+), test on the 150 OASIS-2 subjects ----
# Before running: Add Input -> your 'oasis2-prepped' dataset. Cells 1, 2, 3, 5 and 6 must have been run.
N_SEEDS = 5                                   # 5 independent trainings per view; their probabilities are averaged
s2_csv = sorted(glob.glob(f'{INPUT_ROOT}/**/subjects_oasis2.csv', recursive=True))
n2_files = sorted(glob.glob(f'{INPUT_ROOT}/**/prepped/OAS2_*_MR1.npz', recursive=True))
qc_csv = sorted(glob.glob(f'{INPUT_ROOT}/**/qc.csv', recursive=True))
print('OASIS-2 table:', s2_csv, '| volumes:', len(n2_files), '| qc:', qc_csv)
assert s2_csv and len(n2_files) == 150, 'add the oasis2-prepped dataset to this notebook'
d2 = pd.read_csv(s2_csv[0])
d2['path'] = d2.session_id.map({os.path.basename(f)[:-4]: f for f in n2_files})
assert d2.path.notna().all(), 'some OASIS-2 volumes are missing'
d2['y'] = d2.y_dementia.astype(int); yt = d2.y.values
print('OASIS-2: n =', len(d2), '| dementia', int(yt.sum()), '| controls', int((yt == 0).sum()))

def build_ext(d, mirror):
    X = {v: None for v in CFG['VIEWS']}
    for i, f in enumerate(d.path):
        vol = np.load(f)['vol'].astype(np.float32)
        if mirror: vol = np.ascontiguousarray(vol[::-1])          # left-right mirror (axis 0)
        for v in CFG['VIEWS']:
            s = view_slices(vol, VIEW_AXIS[v], CFG['N_SLICES'], CFG['CENTRAL_FRAC'], CFG['STEP'])
            if X[v] is None: X[v] = np.zeros((len(d),) + s.shape, dtype=np.float16)
            X[v][i] = s
    return X
X2, X2m = build_ext(d2, False), build_ext(d2, True)

d1 = get_task('dementia'); y1 = d1.y.values
idx = subjects_all.reset_index().set_index('subject_id').loc[d1.subject_id, 'index'].values
n1, n2 = len(d1), len(d2)
print('training set (OASIS-1, 60+): n =', n1, '| dementia', int(y1.sum()))

# ---- train on OASIS-1, predict OASIS-2 (the normal and the left-right mirrored copy, by the same models) ----
P = {v: np.zeros((N_SEEDS, 2 * n2)) for v in CFG['VIEWS']}
t0 = time.time()
for v in CFG['VIEWS']:
    Xc = np.concatenate([X_all[v][idx], X2[v], X2m[v]])
    yc = np.concatenate([y1, np.zeros(2 * n2, dtype=int)])        # labels of the test part are never used
    tr, te = np.arange(n1), np.arange(n1, n1 + 2 * n2)
    for s in range(N_SEEDS):
        SAVE_AS = f'{WORK}/weights/{v}_seed{s}.pt'      # keep the trained network for the demo app
        P[v][s] = run_fold_view(Xc, yc, tr, te, seed=5000 + s, epochs=CFG['EPOCHS'])
        SAVE_AS = None
        print(f'  {v} seed {s} done | {time.time()-t0:.0f}s', flush=True)
    del Xc
pv = {v: P[v].mean(0) for v in CFG['VIEWS']}
pens = np.mean([pv[v] for v in CFG['VIEWS']], 0)
p_ens, p_ens_m = pens[:n2], pens[n2:]

# ---- covariate baselines, trained on OASIS-1 and applied to OASIS-2 ----
def cov_pred(cols):
    med = d1[cols].median()
    m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
    m.fit(d1[cols].fillna(med).values.astype(float), y1)
    return m.predict_proba(d2[cols].fillna(med).values.astype(float))[:, 1]
p_age, p_cov = cov_pred(['age']), cov_pred(['age', 'sex', 'educ'])

def row(name, p, thr=True):
    lo, hi = boot_ci(yt, p); r = dict(model=name, auc=roc_auc_score(yt, p), ci_low=lo, ci_high=hi)
    if thr:
        pred = (p >= 0.5).astype(int)
        r['sens'] = ((pred == 1) & (yt == 1)).sum() / (yt == 1).sum(); r['spec'] = ((pred == 0) & (yt == 0)).sum() / (yt == 0).sum()
    return r
rows = [row(v, pv[v][:n2]) for v in CFG['VIEWS']] + [row('ensemble', p_ens)]
rows += [row('age only (no images)', p_age, False), row('age+sex+educ (no images)', p_cov, False)]
tab = pd.DataFrame(rows); tab['bal_acc'] = (tab.sens + tab.spec) / 2
print('\n===== EXTERNAL TEST on OASIS-2 (n=%d) =====' % n2); print(tab.round(3).to_string(index=False))

rng = np.random.default_rng(1); diffs = []
for _ in range(2000):
    i = rng.integers(0, n2, n2)
    if len(set(yt[i])) == 2: diffs.append(roc_auc_score(yt[i], p_ens[i]) - roc_auc_score(yt[i], p_cov[i]))
print('\nAUC gain of images over age+sex+educ: %+.3f (95%% CI %+.3f to %+.3f)' % (np.mean(diffs), *np.percentile(diffs, [2.5, 97.5])))
seed_aucs = [roc_auc_score(yt, np.mean([P[v][s][:n2] for v in CFG['VIEWS']], 0)) for s in range(N_SEEDS)]
print('ensemble AUC for each single seed:', np.round(seed_aucs, 3), '| spread (sd) %.3f' % np.std(seed_aucs))

# ---- sensitivity checks ----
sens_rows = [row('ensemble (as is)', p_ens), row('ensemble, test brains left-right MIRRORED', p_ens_m)]
if qc_csv:
    qc = pd.read_csv(qc_csv[0]).set_index('session_id').reindex(d2.session_id)
    keep = (qc.ncc_to_template >= qc.ncc_to_template.quantile(0.2)).values
    lo, hi = boot_ci(yt[keep], p_ens[keep])
    sens_rows.append(dict(model='ensemble, without the 20%% worst-aligned brains (n=%d)' % keep.sum(), auc=roc_auc_score(yt[keep], p_ens[keep]), ci_low=lo, ci_high=hi))
print('\n--- sensitivity ---'); print(pd.DataFrame(sens_rows).round(3).to_string(index=False))
print('\nmean score: controls %.3f | dementia %.3f' % (p_ens[yt == 0].mean(), p_ens[yt == 1].mean()))

# ---- save + ROC picture ----
out = d2[['subject_id', 'age', 'sex', 'educ', 'CDR', 'y']].copy()
for v in CFG['VIEWS']: out['p_' + v] = pv[v][:n2]
out['p_ensemble'], out['p_ensemble_mirrored'], out['p_age'], out['p_age_sex_educ'] = p_ens, p_ens_m, p_age, p_cov
out.to_csv(f'{WORK}/external_oasis2_predictions.csv', index=False)
tab.round(4).to_csv(f'{WORK}/external_oasis2_results.csv', index=False)
from sklearn.metrics import roc_curve
import matplotlib.pyplot as plt
plt.figure(figsize=(4.5, 4.5))
for nm, p in [('images (ensemble)', p_ens), ('age+sex+educ', p_cov), ('age only', p_age)]:
    fpr, tpr, _ = roc_curve(yt, p); plt.plot(fpr, tpr, label=f'{nm}  AUC {roc_auc_score(yt, p):.2f}')
plt.plot([0, 1], [0, 1], 'k:', lw=0.8); plt.xlabel('1 - specificity'); plt.ylabel('sensitivity'); plt.title('External test: OASIS-2 (n=%d)' % n2)
plt.legend(fontsize=8); plt.tight_layout(); plt.savefig(f'{WORK}/external_roc.png', dpi=150); plt.show()

In [ ]:
# ---- Cell 14: which way is left-right for OASIS-2? Uses NO labels. (run after Cell 13, same session) ----
from scipy.stats import binomtest
t_files = list(subjects_all.path)                  # OASIS-1 volumes
tmpl = np.mean([np.load(f)['vol'].astype(np.float32) for f in t_files[:100]], 0)   # OASIS-1 average from the first 100 only

def ncc_to(v, T):
    m = (v != 0) | (T != 0)
    x, y = v[m] - v[m].mean(), T[m] - T[m].mean()
    return float((x * y).sum() / np.sqrt((x * x).sum() * (y * y).sum()))

def lr_vote(files):
    # positive = the brain as it is matches the OASIS-1 average better than its mirror image
    out = []
    for f in files:
        v = np.load(f)['vol'].astype(np.float32)
        out.append(ncc_to(v, tmpl) - ncc_to(np.ascontiguousarray(v[::-1]), tmpl))
    return np.array(out)

a = lr_vote(t_files[100:])          # OASIS-1 brains that were NOT in the average: the "known correct" control
b = lr_vote(list(d2.path))          # OASIS-2 brains
for name, x in [('OASIS-1 (control, known orientation)', a), ('OASIS-2', b)]:
    k = int((x > 0).sum()); p = binomtest(k, len(x), 0.5).pvalue
    print(f'{name:38s}: as-is matches better in {k}/{len(x)} = {k/len(x):.0%} | mean difference {x.mean():+.4f} | sign-test p = {p:.2g}')
print('\nHow to read it: for OASIS-1 the share should be clearly above 50%. If OASIS-2 shows the same (well above 50%), the OASIS-2 brains are the right way round.')
print('If OASIS-2 is clearly below 50% (and OASIS-1 is above), the OASIS-2 brains are mirrored and the mirrored result is the correct one.')
print('If both are near 50%, this check cannot tell, and we report both versions.')

In [ ]:
# ---- Cell 14b: left-right check, fixed version (no labels). Run after Cell 13 / Cell 14 in the same session. ----
# Fix: the first check mirrored each brain around the middle of the picture, which also moved the brain sideways by ~1.6 voxels.
# Here each mirrored brain is moved back so its centre of mass stays where it was. Only the left-right swap is left.
from scipy import ndimage
from scipy.stats import binomtest

def cx(v):
    return np.argwhere(v != 0)[:, 0].mean()

def mirror_keep_centre(v):
    m = np.ascontiguousarray(v[::-1])
    return ndimage.shift(m, (cx(v) - cx(m), 0, 0), order=1)

def lr_vote2(files):
    out = []
    for f in files:
        v = np.load(f)['vol'].astype(np.float32)
        out.append(ncc_to(v, tmpl) - ncc_to(mirror_keep_centre(v), tmpl))
    return np.array(out)

a2 = lr_vote2(t_files[100:])        # OASIS-1 brains NOT in the average (control: orientation known)
b2 = lr_vote2(list(d2.path))        # OASIS-2
for name, x in [('OASIS-1 (control)', a2), ('OASIS-2', b2)]:
    k = int((x > 0).sum()); p = binomtest(k, len(x), 0.5).pvalue
    print(f'{name:18s}: as-is matches better in {k}/{len(x)} = {k/len(x):.0%} | mean difference {x.mean():+.4f} | sign-test p = {p:.2g}')

# second check: the population-level left-right asymmetry pattern (average of many brains) in both datasets
def asym(files):
    m = np.mean([np.load(f)['vol'].astype(np.float32) for f in files], 0)
    return m - m[::-1]
A1, A2 = asym(t_files), asym(list(d2.path))
msk = (tmpl != 0) & (np.abs(np.arange(tmpl.shape[0])[:, None, None] - tmpl.shape[0] / 2) > 4)
r = np.corrcoef(A1[msk], A2[msk])[0, 1]
print(f'\nleft-right asymmetry pattern, OASIS-1 vs OASIS-2: correlation {r:+.3f}  (positive = same way round, negative = mirrored)')
print('\nRule (fixed now, before looking): the check counts only if the OASIS-1 control is clearly above 50% (65% or more).')
print('Then: OASIS-2 well above 50% -> as-is is right | OASIS-2 well below 50% -> mirrored is right | otherwise: cannot tell, report both.')

In [ ]:
# ---- Cell 15: how much does mirroring hurt on OASIS-1 itself? (about 10 minutes, GPU) ----
# Cells 1, 2, 3, 5, 6 must have been run (60+ run, 198 subjects). Cell 13 gives build_ext(); if you restarted, run Cell 13 first or paste its build_ext function.
# Idea: train on OASIS-1 folds as usual, then test each fold on the normal brains AND on their left-right mirror images.
# If a mirrored test brain loses accuracy here, the model is sensitive to left-right.
# Then the OASIS-2 result (mirroring raised AUC by +0.064) tells us which way round OASIS-2 really is.
d1 = get_task('dementia'); y1 = d1.y.values
idx = subjects_all.reset_index().set_index('subject_id').loc[d1.subject_id, 'index'].values
d1m = d1.copy()
Xm = build_ext(d1, True)                         # mirrored copies of the OASIS-1 brains (same shift as for OASIS-2)
folds = make_folds(d1, 1)                        # repeat 0, 5 subject-level folds
N = len(d1)
Po = {v: np.full(N, np.nan) for v in CFG['VIEWS']}; Pm = {v: np.full(N, np.nan) for v in CFG['VIEWS']}
t0 = time.time()
for r, k, tr, te in folds:
    for v in CFG['VIEWS']:
        Xc = np.concatenate([X_all[v][idx], Xm[v]])
        te_cat = np.concatenate([te, te + N])
        p = run_fold_view(Xc, np.concatenate([y1, y1]), tr, te_cat, seed=7000 + k, epochs=CFG['EPOCHS'])
        Po[v][te], Pm[v][te] = p[:len(te)], p[len(te):]
        del Xc
    print(f'fold {k} done | {time.time()-t0:.0f}s', flush=True)
eo = np.mean([Po[v] for v in CFG['VIEWS']], 0); em = np.mean([Pm[v] for v in CFG['VIEWS']], 0)
print('\nOASIS-1 cross-validation (repeat 0): AUC normal %.3f | AUC mirrored %.3f' % (roc_auc_score(y1, eo), roc_auc_score(y1, em)))
rng = np.random.default_rng(2); df = []
for _ in range(2000):
    i = rng.integers(0, N, N)
    if len(set(y1[i])) == 2: df.append(roc_auc_score(y1[i], em[i]) - roc_auc_score(y1[i], eo[i]))
print('mirrored minus normal: %+.3f (95%% CI %+.3f to %+.3f)' % (np.mean(df), *np.percentile(df, [2.5, 97.5])))
for v in CFG['VIEWS']:
    print(f'   {v:9s} normal {roc_auc_score(y1, Po[v]):.3f} | mirrored {roc_auc_score(y1, Pm[v]):.3f}')
print('\nCompare with OASIS-2, where mirrored minus normal was +0.064 (CI +0.036 to +0.095).')
print('Rule (fixed before looking): if mirroring HURTS on OASIS-1 (CI below zero) and HELPED on OASIS-2, OASIS-2 is most likely mirrored. If mirroring changes nothing on OASIS-1, the OASIS-2 gain is not about orientation.')

In [ ]:
# ---- Cell 16: save the settings the demo app needs, and zip the weights (run after Cell 13) ----
import shutil
vol_shape = list(np.load(d2.path[0])['vol'].shape)
json.dump(dict(views=CFG['VIEWS'], view_axis=VIEW_AXIS, n_slices=CFG['N_SLICES'],
               central_frac=CFG['CENTRAL_FRAC'], step=CFG['STEP'], img=CFG['IMG'],
               seeds=N_SEEDS, volume_shape=vol_shape, threshold=0.5),
          open(f'{WORK}/config.json', 'w'), indent=1)
shutil.make_archive(f'{WORK}/weights', 'zip', f'{WORK}/weights')
print(vol_shape, os.listdir(f'{WORK}/weights'))
